In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip '/content/drive/MyDrive/banana_dataset.zip'

## Rename the image filename and create csv

In [ ]:
#creating csv for training dataset

import os
import pandas as pd

train_dir='/content/Banana Ripeness Classification Dataset/train'
classes=['overripe','ripe','rotten','unripe']

data=[]

for cls in classes:
  class_path=os.path.join(train_dir,cls)
  images=sorted(os.listdir(class_path))
  for i,img_name in enumerate(images,start=1):
    ext=os.path.splitext(img_name)[1]
    new_filename=f"img-{i}_{cls}{ext}"
    new_path=os.path.join(class_path,new_filename)
    old_path=os.path.join(class_path,img_name)
    os.rename(old_path,new_path)
    data.append(
        {
            "filename":new_filename,
            "label":cls,
            "path":new_path
        }

    )

df=pd.DataFrame(data)
csv_path=os.path.join(train_dir,'train.csv')
df.to_csv(csv_path,index=False)
print("CSV created successfully!")

In [ ]:
df.head()

In [ ]:
#creating csv for testing dataset
import os
import pandas as pd

test_dir='/content/Banana Ripeness Classification Dataset/test'
classes=['overripe','ripe','rotten','unripe']

data=[]

for cls in classes:
  class_path=os.path.join(test_dir,cls)
  images=sorted(os.listdir(class_path))
  for i,img_name in enumerate(images,start=1):
    ext=os.path.splitext(img_name)[1]
    new_filename=f"img-{i}_{cls}{ext}"
    new_path=os.path.join(class_path,new_filename)
    old_path=os.path.join(class_path,img_name)
    os.rename(old_path,new_path)
    data.append(
        {
            "filename":new_filename,
            "label":cls,
            "path":new_path
        }

    )

df=pd.DataFrame(data)
csv_path=os.path.join(test_dir,'test.csv')
df.to_csv(csv_path,index=False)
print("CSV created successfully!")


In [ ]:
#creating csv for validation dataset
import os
import pandas as pd

valid_dir='/content/Banana Ripeness Classification Dataset/valid'
classes=['overripe','ripe','rotten','unripe']

data=[]

for cls in classes:
  class_path=os.path.join(valid_dir,cls)
  images=sorted(os.listdir(class_path))
  for i,img_name in enumerate(images,start=1):
    ext=os.path.splitext(img_name)[1]
    new_filename=f"img-{i}_{cls}{ext}"
    new_path=os.path.join(class_path,new_filename)
    old_path=os.path.join(class_path,img_name)
    os.rename(old_path,new_path)
    data.append(
        {
            "filename":new_filename,
            "label":cls,
            "path":new_path
        }

    )

df=pd.DataFrame(data)
csv_path=os.path.join(valid_dir,'valid.csv')
df.to_csv(csv_path,index=False)
print("CSV created successfully!")


In [ ]:
%cd /content

!zip -r banana_final_dataset.zip "Banana Ripeness Classification Dataset"

In [ ]:
!unzip '/content/drive/MyDrive/banana_final_dataset.zip'

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
csv_path='/content/Banana Ripeness Classification Dataset/train/train.csv'
df=pd.read_csv(csv_path)

plt.figure(figsize=(20,4))
for i,row in enumerate(df.sample(5).itertuples(),start=1):
  img=Image.open(row.path)
  plt.subplot(1,5,i)
  wt,ht=img.size
  plt.imshow(img)
  plt.axis('off')
  plt.title(f"h:{ht},w:{wt} ; {row.label}")




In [ ]:
import random
train_df=pd.read_csv('/content/Banana Ripeness Classification Dataset/train/train.csv')
test_df=pd.read_csv('/content/Banana Ripeness Classification Dataset/test/test.csv')
valid_df=pd.read_csv('/content/Banana Ripeness Classification Dataset/valid/valid.csv')

## Load dataset batch  wise

In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet  import preprocess_input
train_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input,
    #Applying data augmentation
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)
train_generator=train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='label',
    target_size=(224,224),#resize
    batch_size=152,
    class_mode='categorical'
)


In [ ]:
valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
)

valid_generator = valid_datagen.flow_from_dataframe(
    dataframe=valid_df,
    x_col='path',
    y_col='label',
    target_size=(224,224),
    batch_size=152,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col='label',
    target_size=(224,224),
    batch_size=152,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense,GlobalAveragePooling2D,Dropout
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze(weights are not updated of EfficientNet layers)
base_model.trainable = False
#customise layers added
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

x = Dropout(0.3)(x)# to reduce dependency , random 30% subset of the 128 neurons is deactivated

output = Dense(4, activation='softmax')(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=10
)

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)

print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

predictions = model.predict(test_generator)

y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print(classification_report(
    y_true,
    y_pred,
    target_names=list(test_generator.class_indices.keys())
))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=list(test_generator.class_indices.keys()),
    yticklabels=list(test_generator.class_indices.keys())
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
model.save('banana_shelf_life_prediction.keras')

In [ ]:
from tensorflow.keras.models import load_model
model=load_model('/content/banana_shelf_life_prediction.keras')

In [ ]:
print(train_generator.class_indices)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input

class_names=['overripe','ripe','rotten','unripe']
def preprocess_img(img_path):
  img=image.load_img(
      img_path,
      target_size=(224,224)
  )
  img_array=image.img_to_array(img)
  img_array=np.expand_dims(img_array,axis=0)
  img_array=preprocess_input(img_array)
  return img_array

In [ ]:
shelf_life = {
    "unripe": "6-9 days",
    "ripe": "3-5 days",
    "overripe": "1-2 days",
    "rotten": "0 days"
}

img_path='inp.jpeg'
pred=model.predict(preprocess_img(img_path))
predicted_class=class_names[np.argmax(pred)]


print(
    "Estimated Shelf Life:",
    shelf_life[predicted_class]
)